# 02 — Clean & Derive language metrics

Load `speeches_raw` from DuckDB, clean it, and derive the per-speech metrics the
lead question needs. Output: `data/interim/speeches_clean.parquet` + a DuckDB
`speeches_clean` table.

**Lead question:** self (I/me/my) vs collective (we/us/our) framing by president,
and the 1789→present trend.

## ⭐ Guarding against apples-to-oranges

The Miller Center corpus is **curated** and coverage is far denser for modern
presidents (LBJ 71 speeches … Garfield 1). So a raw whole-corpus per-president
ranking mixes speech types unevenly. We derive a **`speech_type`** classification so
the analysis can be cut three ways (all validated in exploration; only like-for-like
cuts become social posts):

1. **Whole corpus** — everything; exploration only, not a headline comparison.
2. **State of the Union** — the broadest apples-to-apples series. The Article II
   annual address to Congress was labeled *Annual Message* through 1928 and *State
   of the Union* from 1929 on — the **same institutional speech**, unified here as
   `is_sotu_series` (~209 speeches, ~40 presidents, 1790→present).
3. **Inaugural Address** — the other clean lens: 58 speeches, 39 presidents,
   1789→present, one tightly-defined occasion per president.

**Series break to carry forward:** even within the SOTU series there is a
**written-vs-spoken** break — Jefferson→Taft *submitted written messages* (read by a
clerk); delivery as spoken oratory resumed with Wilson in 1913. Written messages run
long and formal and use pronouns differently than delivered speeches, so we flag
`delivery_mode` and treat pre/post-1913 as distinct within any SOTU trend.

Metric logic lives in `src/clean_quality.py` (`add_speech_metrics`, `classify_speech_type`);
tokenization is a transparent lowercase word regex (contractions like *I'm* / *we'll*
/ *let's* attributed to their pronoun) so every count is explainable in the codebook.

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.ingest import load_config
from src.clean_quality import (
    get_connection, run_sql, quality_report, save_interim,
    load_to_duckdb, add_speech_metrics,
)

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')

## Load raw + parse date/year in DuckDB

Read `speeches_raw` and keep only the columns we use (drop `transcript_html`,
`introduction`, `video`, `audio`, `uuid`). The `date` field is ISO with an odd
fixed `-04:56` offset on every row — an artifact, not a real timezone — so we take
the leading `YYYY-MM-DD` and derive `year` from it.

In [ ]:
df = run_sql("""
    SELECT
        TRIM(president)              AS president,
        CAST(date[1:10] AS DATE)     AS speech_date,
        CAST(date[1:4]  AS INTEGER)  AS year,
        TRIM(title)                  AS title,
        transcript,
        url,
        source_file
    FROM speeches_raw
    WHERE transcript IS NOT NULL AND TRIM(transcript) <> ''
""", con)
print(df.shape)
print('year span:', int(df.year.min()), '→', int(df.year.max()))
df.head(3)

## Derive language metrics + speech type

`add_speech_metrics()` adds per speech: `word_count`, `self_count`,
`collective_count`, `self_per_1k`, `collective_per_1k`, `self_share`
(= self / (self+collective)), `speech_type`, and `is_sotu_series`. Rates are per
1,000 words so long and short speeches are comparable.

In [ ]:
df = add_speech_metrics(df, text_col='transcript', title_col='title')

# delivery_mode: within the SOTU series, written messages (Jefferson→Taft) vs
# spoken addresses (Washington/Adams, then Wilson 1913 onward). Applied to the
# whole frame; only load-bearing for the SOTU lens.
df['delivery_mode'] = df['year'].map(lambda y: 'written_era' if 1801 <= y <= 1912 else 'spoken_era')

df[['president','year','speech_type','word_count','self_count','collective_count',
    'self_per_1k','collective_per_1k','self_share']].head(8)

## Sanity-check the classification + the three lenses

Confirm the speech-type buckets, and that the two clean comparison lenses (SOTU
series, Inaugurals) have the broad presidential coverage we expect — vs the whole
corpus, whose per-president counts are a coverage artifact.

In [ ]:
print('=== speech_type counts ===')
print(df['speech_type'].value_counts().to_string())

print('\n=== coverage by lens (n speeches, n presidents, year span) ===')
for lens, mask in {
    'Whole corpus':        df['president'].notna(),
    'SOTU series':         df['is_sotu_series'],
    'Inaugural Address':   df['speech_type'] == 'Inaugural Address',
}.items():
    sub = df[mask]
    print(f"{lens:20s} n={len(sub):4d}  presidents={sub['president'].nunique():3d}  "
          f"{int(sub.year.min())}–{int(sub.year.max())}")

print('\n=== SOTU series: written vs spoken era ===')
print(df[df.is_sotu_series].groupby('delivery_mode')
      .agg(n=('title','size'), presidents=('president','nunique'),
           yr_min=('year','min'), yr_max=('year','max')).to_string())

## Quality report

In [ ]:
qr = quality_report(
    df,
    table_name='speeches_clean',
    con=con,
    required_columns=['president','year','speech_type','word_count',
                      'self_count','collective_count'],
    max_null_pct=0.05,
)
# self_share is intentionally null when a speech has zero self+collective pronouns
# (rare, very short texts) — that is expected, not a data error.
print('\nspeeches with 0 self+collective pronouns (self_share null):',
      int((df.self_count + df.collective_count == 0).sum()))

## Save interim (Parquet + DuckDB table)

Persist the cleaned, metric-enriched frame to `data/interim/` as Parquet and as the
DuckDB `speeches_clean` table for `03-prepare` / `04-viz`.

In [ ]:
load_to_duckdb(df, 'speeches_clean', con)
save_interim(df, cfg, 'speeches_clean.parquet')
print('speeches_clean rows in DuckDB:',
      con.execute('SELECT COUNT(*) FROM speeches_clean').fetchone()[0])

---
**Next:** `04-viz.ipynb` — explore the self/collective cuts across the three lenses
(whole corpus, SOTU series, inaugurals), per-president and over time, and decide with
the owner which framing(s) are honest enough to become social posts. (`03-prepare`
packages the export once the framing is settled.)

---
## Cleanup
Close the DuckDB connection so the lock is released for other tools (DBCode, other notebooks). Runs on “Run All”.

In [ ]:
con.close()
print('connection closed')